In [ ]:
!pip install -q kagglehub scikit-image

In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

In [ ]:
import kagglehub
path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print(path)

In [ ]:
# collect all images from train/val/test folders, we re-split later
classes = ['NORMAL', 'PNEUMONIA']
files, labels = [], []
for i, c in enumerate(classes):
    f = sorted(glob.glob(os.path.join(path, 'chest_xray', '*', c, '*.jpeg')))
    files += f
    labels += [i] * len(f)
labels = np.array(labels)
print('Total images:', len(files))

In [ ]:
counts = pd.Series(labels).map(dict(enumerate(classes))).value_counts()
print(counts)
sns.barplot(x=counts.index, y=counts.values)
plt.title('Class balance')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, c in enumerate(classes):
    idx = random.sample(list(np.where(labels == i)[0]), 5)
    for j, k in enumerate(idx):
        axes[i, j].imshow(cv2.imread(files[k], cv2.IMREAD_GRAYSCALE), cmap='gray')
        axes[i, j].set_title(c)
        axes[i, j].axis('off')
plt.show()

In [ ]:
# resize to 128x128 grayscale
IMG = 128
X = np.array([cv2.resize(cv2.imread(f, cv2.IMREAD_GRAYSCALE), (IMG, IMG)) for f in files])
y = labels
print(X.shape)

In [ ]:
def extract_features(img):
    h = hog(img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
    lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    glcm = graycomatrix(img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
    g = [graycoprops(glcm, p).mean() for p in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']]
    return np.hstack([h, lbp_hist, g])

feats = np.array([extract_features(img) for img in X])
print('Feature vector shape:', feats.shape)

In [ ]:
img = X[0]
_, hog_img = hog(img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)
lbp_img = local_binary_pattern(img, P=8, R=1, method='uniform')
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [img, hog_img, lbp_img], ['Original', 'HOG', 'LBP']):
    a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()

In [ ]:
# 70 / 15 / 15 split
idx = np.arange(len(X))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=42)
va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=42)
print(len(tr), len(va), len(te))

Xc = X[..., None] / 255.0
X_tr, X_va, X_te = Xc[tr], Xc[va], Xc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]

In [ ]:
def build_cnn(lr=1e-3, dropout=0.3):
    m = models.Sequential([
        layers.Input((IMG, IMG, 1)),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu'), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return m

build_cnn().summary()

In [ ]:
# hyperparameter tuning with 3-fold cross-validation on training set
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = {}
for lr in [1e-3, 1e-4]:
    for dp in [0.3, 0.5]:
        scores = []
        for a, b in skf.split(X_tr, y_tr):
            m = build_cnn(lr, dp)
            m.fit(X_tr[a], y_tr[a], epochs=3, batch_size=32, verbose=0)
            scores.append(m.evaluate(X_tr[b], y_tr[b], verbose=0)[1])
        cv_results[(lr, dp)] = np.mean(scores)
        print(f'lr={lr}, dropout={dp} -> CV acc={np.mean(scores):.4f}')

best_lr, best_dp = max(cv_results, key=cv_results.get)
print('Best:', best_lr, best_dp)

In [ ]:
# class weights because pneumonia images are ~3x normal
cw = {0: len(y_tr) / (2 * np.sum(y_tr == 0)), 1: len(y_tr) / (2 * np.sum(y_tr == 1))}
cnn = build_cnn(best_lr, best_dp)
es = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
hist = cnn.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=15, batch_size=32, class_weight=cw, callbacks=[es])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history['accuracy'], label='train'); ax[0].plot(hist.history['val_accuracy'], label='val'); ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(hist.history['loss'], label='train'); ax[1].plot(hist.history['val_loss'], label='val'); ax[1].set_title('Loss'); ax[1].legend()
plt.show()

In [ ]:
def get_metrics(name, y_true, prob):
    pred = (prob >= 0.5).astype(int)
    return {'Model': name, 'Accuracy': accuracy_score(y_true, pred), 'Precision': precision_score(y_true, pred),
            'Recall': recall_score(y_true, pred), 'F1': f1_score(y_true, pred), 'AUROC': roc_auc_score(y_true, prob)}

cnn_prob = cnn.predict(X_te).ravel()
print(classification_report(y_te, (cnn_prob >= 0.5).astype(int), target_names=classes))

In [ ]:
# classical classifiers on HOG + LBP + GLCM features, tuned with GridSearchCV
sc = StandardScaler()
F_tr, F_te = sc.fit_transform(feats[tr]), sc.transform(feats[te])

svm = GridSearchCV(SVC(probability=True), {'C': [1, 10], 'kernel': ['rbf', 'linear']}, cv=3, n_jobs=-1)
svm.fit(F_tr, y_tr)
print('SVM best:', svm.best_params_)

rf = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [None, 20]}, cv=3, n_jobs=-1)
rf.fit(F_tr, y_tr)
print('RF best:', rf.best_params_)

svm_prob = svm.predict_proba(F_te)[:, 1]
rf_prob = rf.predict_proba(F_te)[:, 1]

In [ ]:
probs = {'CNN': cnn_prob, 'SVM (features)': svm_prob, 'RF (features)': rf_prob}
res = pd.DataFrame([get_metrics(k, y_te, p) for k, p in probs.items()]).set_index('Model')
res.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (name, p) in zip(ax, probs.items()):
    sns.heatmap(confusion_matrix(y_te, (p >= 0.5).astype(int)), annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=a)
    a.set_title(name); a.set_xlabel('Predicted'); a.set_ylabel('Actual')
plt.show()

In [ ]:
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_te, p)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_te, p):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curves'); plt.legend()
plt.show()

In [ ]:
# misclassified test images from CNN
pred = (cnn_prob >= 0.5).astype(int)
wrong = np.where(pred != y_te)[0]
print('False positives (Normal -> Pneumonia):', np.sum((pred == 1) & (y_te == 0)))
print('False negatives (Pneumonia -> Normal):', np.sum((pred == 0) & (y_te == 1)))
if len(wrong):
    fig, axes = plt.subplots(1, min(6, len(wrong)), figsize=(15, 3), squeeze=False)
    for a, k in zip(axes[0], wrong[:6]):
        a.imshow(X_te[k].squeeze(), cmap='gray')
        a.set_title(f'True:{classes[y_te[k]]}\nPred:{classes[pred[k]]}', fontsize=9); a.axis('off')
    plt.show()

In [ ]:
print('Best model by F1:', res['F1'].idxmax())